# RAIL lung-nodule counting — Colab setup

Before running anything:
1. **Runtime > Change runtime type > GPU** (T4 is fine).
2. Upload `lidc_idri_p1-20.zip` to your Google Drive. This pipeline only ever uses patients 1-20 (every script defaults to `--start 1 --end 20`), so you don't need the full 99-patient/11GB `LDIC-IDRI-subset` — just those 20 patients + `annotations.csv`, zipped to ~1.1GB. (If you need more patients later, re-zip a wider range the same way.)
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face access token that has accepted the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/freya-gul/rail.git
%cd rail

Point this at wherever you uploaded `lidc_idri_p1-20.zip` in Drive. It unzips into `datasets/LDIC-IDRI-subset/` inside the cloned repo — that's the path every script's `DICOM_ROOT` already expects, so nothing else needs to change:

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/lidc_idri_p1-20.zip"  # <-- update to your actual upload path
DATA_DIR = "datasets/LDIC-IDRI-subset"  # relative to the repo root (we've already %cd'd into rail)

import pathlib
assert pathlib.Path(ZIP_PATH).exists(), f"{ZIP_PATH} not found — check the path/upload"

In [ ]:
!mkdir -p {DATA_DIR}
!unzip -q {ZIP_PATH} -d {DATA_DIR}
!ls {DATA_DIR}

In [ ]:
!pip install -q -r requirements-colab.txt

pylidc keeps its own scan index (built by scanning the configured DICOM directory the first time it's queried) and reads its config from `~/.pylidcrc`, not the repo's copy — write one pointing at the Drive path:

In [ ]:
import pathlib

pylidcrc = pathlib.Path.home() / ".pylidcrc"
abs_dicom_path = pathlib.Path(DATA_DIR).resolve() / "lidc_idri"
pylidcrc.write_text(f"[dicom]\npath = {abs_dicom_path}\nwarn = True\n")
print(pylidcrc.read_text())

Sanity check: GPU visible to torch, and the detector/MedGemma scripts will now pick it (they already default to `cuda` > `mps` > `cpu`):

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Run the pipeline

`ground_truth_annotations.json` and `detection_cache/` are already committed to the repo for patients 1-20, so the two cells below only need to run if you want to regenerate them or extend the patient range.

In [ ]:
# Ground truth (pylidc) — CPU-only, only needed to regenerate/extend beyond patients 1-20
!python image_download/6_ground_truth_annotations.py --start 1 --end 20

In [ ]:
# MONAI RetinaNet detection + FROC evaluation vs. ground truth (GPU)
!python image_download/7_evaluate_detection.py --start 1 --end 20

In [ ]:
# MedGemma-only whole-volume counting baseline vs. ground truth (GPU)
!python image_download/8_evaluate_medgemma_counting.py --start 1 --end 20

Results land back in the repo: `image_download/detection_froc.csv`, `image_download/detection_counts.json`, `image_download/medgemma_counting_comparison.csv`. Both eval scripts cache per-patient work (`detection_cache/`, `medgemma_count_cache/`) so a run can be safely interrupted and resumed.